# LMP-SPARK
**Author:** Ryan J. McLaughlin  
**Date:** 2025-05-06

This is meant to be a complete end-to-end document for running:

1. Amplicon Sequence Variant (ASV) pipeline
2. General statistics
3. Downstream analytics
4. Figure/Table creation

## 1. Amplicon Sequence Variant (ASV) pipeline
This section reviews the steps involved in creating ASVs from raw FASTQ data.

### Setup Environments for running the pipeline

In [ ]:
%%bash
# Define the environment name
ENV_NAME="spark_env"
ENV_YAML="$PWD/spark_env.yaml"

QI_NAME="qiime2-amplicon-2024.10"

# Check if the environment exists
if mamba env list | grep -q "^${ENV_NAME} "; then
    echo "Environment ${ENV_NAME} already exists."
else
    echo "Environment ${ENV_NAME} does not exist. Creating it..."
    mamba env create -y -n ${ENV_NAME} -f ${ENV_YAML}
fi

# Check if the QIIME2 environment exists
if mamba env list | grep -q "^${QI_NAME} "; then
    echo "Environment ${QI_NAME} already exists."
else
    echo "Environment ${QI_NAME} does not exist. Creating it..."
    mamba env create -y -n ${QI_NAME} -c bioconda qiime2-amplicon-2024.10
fi

### Run the ASV pipeline

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

./run_vsearch.sh

### Run General Statistics

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env
THREADS="$(nproc)"

seqkit stat -a -T -o ../spark_old_output/stats/fastq_stats.tsv -j ${THREADS} ../spark_old_output/fastq_input/*.fastq.gz
seqkit stat -a -T -o ../spark_old_output/stats/fastp_fastqs.tsv -j ${THREADS} ../spark_old_output/fastp/*.fastq.gz
seqkit stat -a -T -o ../spark_old_output/stats/filtered_fastqs.tsv -j ${THREADS} ../spark_old_output/filtered/*.fasta
seqkit stat -a -T -o ../spark_old_output/stats/concat_fastas.tsv -j ${THREADS} ../spark_old_output/concat/concat.fasta

### Run QIIME2 Taxonomic Classifier

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate qiime2-amplicon-2024.10
mkdir -p ../spark_old_output/taxonomy
awk '/^>/ {print; next} {print toupper($0)}' ../spark_old_output/ASVs/ASVs_filtered.fasta > ../spark_old_output/ASVs/ASVs.upper.fasta
python qiime_vs_classifier.py \
  --input-fasta ../spark_old_output/ASVs/ASVs.upper.fasta \
  --ref-taxonomy ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/silva-138_2-ssu-nr99-tax.qza \
  --ref-seqs ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/silva-138_2-ssu-nr99-seqs-DNA.qza \
  --output-tsv ../spark_old_output/taxonomy/ASV_SILVA_tax.full-length.vsearch.tsv \
  --stats-output ../spark_old_output/taxonomy/ASV_SILVA_stats.full-length.vsearch.tsv

### Mitomaster, decontamination, mitoDB

In [ ]:
rm -rf ../spark_old_output/mito
mkdir -p ../spark_old_output/mito/mitomap
rm -rf ../spark_old_output/ASVs/chunks
seqkit split -s 10 -O ../spark_old_output/ASVs/chunks ../spark_old_output/ASVs/ASVs_filtered.fasta
python ./mitomaster.py
blastn -query ~/SeqData/SeqData/UBC/LMP_priority1/spark_old_output/ASVs/ASVs_filtered.fasta \
       -db ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/mito_ncbi \
       -outfmt "6 qseqid sseqid pident length qlen mismatch gapopen qstart qend sstart send evalue bitscore" \
       -out ~/SeqData/SeqData/UBC/LMP_priority1/spark_old_output/mito/mitomap/mito_ncbi.blast6.tsv
blastn -query ~/SeqData/SeqData/UBC/LMP_priority1/spark_old_output/ASVs/ASVs_filtered.fasta \
       -db ~/SeqData/SeqData/UBC/LMP/SPARK_data/ref_db/ssu_pipeline_contaminants \
       -outfmt "6 qseqid sseqid pident length qlen mismatch gapopen qstart qend sstart send evalue bitscore" \
       -out ~/SeqData/SeqData/UBC/LMP_priority1/spark_old_output/mito/mitomap/ssu_pipeline_contaminants.blast6.tsv
python ./mito_checker.py

### Filter ASV count tables

In [ ]:
mkdir -p ../spark_old_output/mito/ASVs
python filter_nontarget.py \
    ~/SeqData/SeqData/UBC/LMP_priority1/spark_old_output/ASVs/ASV_filtered.tsv \
    ~/SeqData/SeqData/UBC/LMP_priority1/spark_old_output/mito/mitomap/nontarget.master.tsv \
    ~/SeqData/SeqData/UBC/LMP_priority1/spark_old_output/ASVs/ASV_target.tsv \
    0.005

### Build Sankey Diagram

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

mkdir -p ../spark_old_output/metadata
mkdir -p ../spark_old_output/mito/metadata

python sankey_builder.py

### Build Sankey Diagram - BRUSH

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

mkdir -p ../spark_old_output/brush/metadata

python sankey_builder_brush.py

### Plot Metadata

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_metadata_brush.py
python outlier_checker_brush.py

python collectors_curve.py \
    --counts ../spark_old_output/brush/ASVs/ASV_final.micro.tsv \
    --meta ../spark_old_output/brush/metadata/metadata_updated.tsv \
    --sample-id-col sample \
    --group-col subclass2 \
    --out_prefix ../spark_old_output/brush/metadata/collectors_curve \
    --permutations 999 --seed 42 \
    --group-colors "ca-lung=#009E73,ca-contra=#0072B2,ctrl-brush=#6A3D9A" \
    --group-order "ctrl-brush,ca-contra,ca-lung"

### Plot Upset

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_upset_brush.py
python venn_bubbles_brush.py

### Run Alpha and Beta Diversity

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python calc_div_brush.py
python plot_diversity_brush.py

### Run indicspecies (R)

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env
Rscript run_indicspecies_brush.R

### Plot indicspecies Results

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_indicspecies_brush.py

### Plot Clustermaps

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python plot_clustermaps_brush.py

### Run SPIEC-EASI (R)

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

Rscript run_spieceasi_brush.R

### Graph Network

In [ ]:
%%bash
MDIR=$(dirname $(which mamba))
source ${MDIR}/../etc/profile.d/conda.sh
conda activate spark_env

python graph_network_brush.py